# SCD2 History — Day-2 Acceptance

Chạy sau khi Bronze và Silver đã hoàn thành cho `2026-01-01`. Mỗi assertion là một điều kiện nghiệm thu, không phải query minh họa.

In [ ]:
DAY_1 = "2025-12-31"
DAY_2 = "2026-01-01"
DIMENSIONS = {
    "dim_branch": ("branch_code", "branch_sk"),
    "dim_product": ("product_code", "product_sk"),
    "dim_customer": ("customer_id", "customer_sk"),
    "dim_account": ("account_id", "account_sk"),
}
TEST_KEYS = {
    "dim_branch": "'BDG001'",
    "dim_product": "'CASA002'",
    "dim_customer": "10001",
    "dim_account": "100001",
}

try:
    spark
except NameError as exc:
    raise RuntimeError("Hãy chọn kernel 'PySpark (Lakehouse)' trước khi chạy") from exc

## 1. Generic history invariants

In [ ]:
def assert_zero(sql, label):
    count = spark.sql(f"SELECT count(*) AS violations FROM ({sql}) checks").first().violations
    assert count == 0, f"{label}: {count} violation(s)"

for table, (business_key, sk_column) in DIMENSIONS.items():
    target = f"lakehouse.silver.{table}"
    assert_zero(
        f"SELECT {business_key} FROM {target} WHERE is_current=1 GROUP BY {business_key} HAVING count(*) > 1",
        f"{table}: multiple current rows",
    )
    assert_zero(
        f"SELECT {sk_column} FROM {target} GROUP BY {sk_column} HAVING count(*) > 1",
        f"{table}: duplicate SK",
    )
    assert_zero(
        f"SELECT * FROM {target} WHERE effective_from > effective_to OR (is_current=1 AND effective_to <> DATE '9999-12-31') OR (is_current=0 AND effective_to = DATE '9999-12-31')",
        f"{table}: invalid effective range",
    )
    assert_zero(
        f"""
        SELECT * FROM (
            SELECT {business_key}, effective_from,
                   lag(effective_to) OVER (PARTITION BY {business_key} ORDER BY effective_from, {sk_column}) AS previous_to
            FROM {target}
        ) x WHERE previous_to IS NOT NULL AND effective_from <= previous_to
        """,
        f"{table}: overlapping versions",
    )
    print(f"{table}: PASS")

## 2. Controlled keys must have old and new versions

In [ ]:
for table, (business_key, sk_column) in DIMENSIONS.items():
    key_value = TEST_KEYS[table]
    history = spark.sql(f"""
        SELECT *, {sk_column} AS acceptance_sk
        FROM lakehouse.silver.{table}
        WHERE {business_key} = {key_value}
        ORDER BY effective_from
    """)
    print(f"--- {table} / {key_value} ---")
    history.show(truncate=False, vertical=True)
    rows = history.select("effective_from", "effective_to", "is_current", "acceptance_sk").collect()
    assert len(rows) == 2, f"{table} expected 2 versions, found {len(rows)}"
    assert str(rows[0].effective_from) == DAY_1 and str(rows[0].effective_to) == DAY_1
    assert rows[0].is_current == 0
    assert str(rows[1].effective_from) == DAY_2 and str(rows[1].effective_to) == "9999-12-31"
    assert rows[1].is_current == 1
    assert rows[0].acceptance_sk != rows[1].acceptance_sk

## 3. Technical-only change must not create history

Customer `10002` chỉ đổi `last_updated` trong source.

In [ ]:
technical_only = spark.sql("""
    SELECT customer_id, count(*) AS versions, sum(CASE WHEN is_current=1 THEN 1 ELSE 0 END) AS current_versions
    FROM lakehouse.silver.dim_customer
    WHERE customer_id = 10002
    GROUP BY customer_id
""").first()
assert technical_only.versions == 1, technical_only
assert technical_only.current_versions == 1, technical_only
print("Technical-only change: PASS")

## 4. Idempotency fingerprint

Chạy cell đầu để chụp fingerprint, rerun `silver_all_dag` với `cob_dt=2026-01-01`, rồi chạy cell so sánh.

In [ ]:
def dimension_fingerprint(table, sk_column):
    rows = spark.sql(f"SELECT {sk_column} FROM lakehouse.silver.{table} ORDER BY {sk_column}").collect()
    return len(rows), tuple(row[0] for row in rows)

before_rerun = {
    table: dimension_fingerprint(table, sk_column)
    for table, (_, sk_column) in DIMENSIONS.items()
}
{table: fingerprint[0] for table, fingerprint in before_rerun.items()}

In [ ]:
after_rerun = {
    table: dimension_fingerprint(table, sk_column)
    for table, (_, sk_column) in DIMENSIONS.items()
}
assert after_rerun == before_rerun, "Rerun changed row counts or surrogate-key sets"
print("IDEMPOTENCY ACCEPTED")